In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install roboflow
import os
os.makedirs("/content/dataset_local", exist_ok=True)
os.chdir("/content/dataset_local")

from roboflow import Roboflow
rf=Roboflow(api_key="qIfhltEADLm5w4R791Kd")
project=rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version=project.version(12)
dataset=version.download("yolov8")
print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-12 in yolov8:: 100%|██████████| 20262/20262 [00:02<00:00, 6899.56it/s]


/content/dataset_local/License-Plate-Recognition-12


In [3]:
import cv2
import numpy as np
import pandas as pd
import random
import shutil
import os

extractPath=dataset.location
imgDir=f"{extractPath}/train/images"
labelDir=f"{extractPath}/train/labels"

records=[]
for fname in os.listdir(imgDir):
    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        img=cv2.imread(os.path.join(imgDir, fname))
        if img is None:
          continue
        gray=cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness=np.mean(gray)
        blur=cv2.Laplacian(gray, cv2.CV_64F).var()
        if brightness<90:
          dayNight="night"
        else:
          dayNight="day"
        if blur<100:
          sharpness="blurry"
        else:
          sharpness="sharp"
        records.append({"filename": fname, "combined": f"{dayNight}_{sharpness}"})

df=pd.DataFrame(records)
print("--- Condition Distribution ---")
print(df['combined'].value_counts())

df.to_csv("/content/drive/MyDrive/license_plate_project/dataset/condition_report.csv", index=False)
print("Report saved to Drive!")

balancedImgDir=f"{extractPath}/train_balanced/images"
balancedLabelDir=f"{extractPath}/train_balanced/labels"
os.makedirs(balancedImgDir, exist_ok=True)
os.makedirs(balancedLabelDir, exist_ok=True)
targets={"day_sharp": 3500, "night_sharp": 3500, "day_blurry": 500, "night_blurry": 500}
for combo, targetCount in targets.items():
    subset=df[df['combined']==combo]['filename'].tolist()
    currCount=len(subset)
    print(f"{combo}: current={currCount}, target={targetCount}")
    if currCount==0:
      continue
    if currCount>targetCount:
      selected=random.sample(subset, targetCount)
      for fname in selected:
        base=os.path.splitext(fname)[0]
        shutil.copy(os.path.join(imgDir, fname), os.path.join(balancedImgDir, fname))
        shutil.copy(os.path.join(labelDir, base+".txt"), os.path.join(balancedLabelDir, base+".txt"))
    else:
      for fname in subset:
        base=os.path.splitext(fname)[0]
        shutil.copy(os.path.join(imgDir, fname), os.path.join(balancedImgDir, fname))

--- Condition Distribution ---
combined
day_sharp       4916
night_sharp     2056
day_blurry        54
night_blurry      31
Name: count, dtype: int64
Report saved to Drive!
day_sharp: current=4916, target=3500
night_sharp: current=2056, target=3500
day_blurry: current=54, target=500
night_blurry: current=31, target=500


In [4]:
balancedImgDir=f"{dataset.location}/train_balanced/images"
import os
print(len(os.listdir(balancedImgDir)))

5641


In [5]:
import shutil, os
balancedDir=f"{dataset.location}/train_balanced"
if os.path.exists(balancedDir):
  shutil.rmtree(balancedDir)
  print("Old balanced folder deleted")

Old balanced folder deleted


In [6]:
import random
extractPath=dataset.location
imgDir=f"{extractPath}/train/images"
labelDir=f"{extractPath}/train/labels"

balancedImgDir=f"{extractPath}/train_balanced/images"
balancedLabelDir=f"{extractPath}/train_balanced/labels"
os.makedirs(balancedImgDir, exist_ok=True)
os.makedirs(balancedLabelDir, exist_ok=True)
targets={"day_sharp": 3500, "night_sharp": 3500, "day_blurry": 500, "night_blurry": 500}
for combo, targetCount in targets.items():
  subset=df[df['combined']==combo]['filename'].tolist()
  currCount=len(subset)
  print(f"Processing {combo}: current={currCount}, target={targetCount}")
  copied=0
  try:
    if currCount>targetCount:
      selected=random.sample(subset,targetCount)
      for fname in selected:
        base=os.path.splitext(fname)[0]
        shutil.copy(os.path.join(imgDir, fname), os.path.join(balancedImgDir, fname))
        shutil.copy(os.path.join(labelDir, base+".txt"), os.path.join(balancedLabelDir, base+".txt"))
        copied+=1
    else:
      for fname in subset:
        base=os.path.splitext(fname)[0]
        shutil.copy(os.path.join(imgDir, fname), os.path.join(balancedImgDir, fname))
        shutil.copy(os.path.join(labelDir, base+".txt"), os.path.join(balancedLabelDir, base+".txt"))
        copied+=1
      needed=targetCount-currCount
      for i in range(needed):
        fname=random.choice(subset)
        base=os.path.splitext(fname)[0]
        newName=f"{base}_dup{i}"
        shutil.copy(os.path.join(imgDir, fname), os.path.join(balancedImgDir, newName+".jpg"))
        shutil.copy(os.path.join(labelDir, base+".txt"), os.path.join(balancedLabelDir, newName+".txt"))
        copied+=1
  except Exception as e:
    print(f"ERROR in {combo}: {e}")
  print(f"  -> Copied {copied} files for {combo}")
print("\n=== Balancing done! ===")
finalCount = len([f for f in os.listdir(balancedImgDir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f"Total balanced training images: {finalCount}")

Processing day_sharp: current=4916, target=3500
  -> Copied 3500 files for day_sharp
Processing night_sharp: current=2056, target=3500
  -> Copied 3500 files for night_sharp
Processing day_blurry: current=54, target=500
  -> Copied 500 files for day_blurry
Processing night_blurry: current=31, target=500
  -> Copied 500 files for night_blurry

=== Balancing done! ===
Total balanced training images: 8000


In [7]:
!pip install albumentations

In [8]:
import albumentations as A
import cv2
import os

extractPath = dataset.location
balancedImgDir = f"{extractPath}/train_balanced/images"
balancedLabelDir = f"{extractPath}/train_balanced/labels"

augImgDir = f"{extractPath}/train_augmented/images"
augLabelDir = f"{extractPath}/train_augmented/labels"
os.makedirs(augImgDir, exist_ok=True)
os.makedirs(augLabelDir, exist_ok=True)

transform = A.Compose([
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.RandomCrop(width=480, height=480, p=0.3),
    A.CoarseDropout(max_holes=3, max_height=40, max_width=40, p=0.4),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def load_yolo_labels(label_path):
    boxes, classes = [], []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls, x, y, w, h = map(float, parts)
            boxes.append([x, y, w, h])
            classes.append(int(cls))
    return boxes, classes

def clip_box(box):
    x, y, w, h = box
    return [min(max(x,0.0),1.0), min(max(y,0.0),1.0), min(max(w,0.0),1.0), min(max(h,0.0),1.0)]

count_success = 0
count_skipped = 0

all_files = [f for f in os.listdir(balancedImgDir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
total_files = len(all_files)
print(f"Total files to process: {total_files}")

for idx, fname in enumerate(all_files):
    base = os.path.splitext(fname)[0]
    img_path = os.path.join(balancedImgDir, fname)
    label_path = os.path.join(balancedLabelDir, base + ".txt")

    if not os.path.exists(label_path):
        count_skipped += 1
        continue

    image = cv2.imread(img_path)
    if image is None:
        count_skipped += 1
        continue

    boxes, classes = load_yolo_labels(label_path)
    if len(boxes) == 0:
        count_skipped += 1
        continue

    cv2.imwrite(os.path.join(augImgDir, fname), image)
    with open(label_path) as f:
        label_content = f.read()
    with open(os.path.join(augLabelDir, base + ".txt"), 'w') as f:
        f.write(label_content)

    try:
        boxes_clipped = [clip_box(b) for b in boxes]
        augmented = transform(image=image, bboxes=boxes_clipped, class_labels=classes)
        aug_img = augmented['image']
        aug_boxes = augmented['bboxes']
        aug_classes = augmented['class_labels']

        if len(aug_boxes) == 0:
            count_skipped += 1
            continue

        new_name = f"{base}_aug"
        cv2.imwrite(os.path.join(augImgDir, new_name + ".jpg"), aug_img)

        with open(os.path.join(augLabelDir, new_name + ".txt"), 'w') as f:
            for cls, box in zip(aug_classes, aug_boxes):
                f.write(f"{cls} {' '.join(map(str, box))}\n")

        count_success += 1
    except Exception as e:
        count_skipped += 1

    if idx % 1000 == 0:
        print(f"Processed {idx}/{total_files}...")

print(f"\nDone! Successfully augmented: {count_success}, Skipped: {count_skipped}")
final_count = len([f for f in os.listdir(augImgDir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f"Total images in augmented folder: {final_count}")

Argument(s) 'var_limit' are not valid for transform GaussNoise
Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout


Total files to process: 8000
Processed 0/8000...


invalid value encountered in divide


Processed 1000/8000...
Processed 2000/8000...
Processed 3000/8000...
Processed 4000/8000...
Processed 5000/8000...
Processed 6000/8000...
Processed 7000/8000...

Done! Successfully augmented: 5900, Skipped: 2100
Total images in augmented folder: 13875


In [9]:
extractPath = dataset.location

yaml_content = f"""train: {extractPath}/train_augmented/images
val: {extractPath}/valid/images
test: {extractPath}/test/images

nc: 1
names: ['License_Plate']
"""

with open(f"{extractPath}/data_final.yaml", "w") as f:
    f.write(yaml_content)

print("data_final.yaml created!")
print(yaml_content)

data_final.yaml created!
train: /content/dataset_local/License-Plate-Recognition-12/train_augmented/images
val: /content/dataset_local/License-Plate-Recognition-12/valid/images
test: /content/dataset_local/License-Plate-Recognition-12/test/images

nc: 1
names: ['License_Plate']



In [10]:
!pip install ultralytics

from ultralytics import YOLO
import shutil
import os

model = YOLO("yolov8n.pt")

DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/license_plate_project/weights"
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)

def save_to_drive_callback(trainer):
    save_dir = trainer.save_dir
    best_path = os.path.join(save_dir, "weights", "best.pt")
    last_path = os.path.join(save_dir, "weights", "last.pt")

    if os.path.exists(best_path):
        shutil.copy(best_path, os.path.join(DRIVE_WEIGHTS_DIR, "best.pt"))
    if os.path.exists(last_path):
        shutil.copy(last_path, os.path.join(DRIVE_WEIGHTS_DIR, "last.pt"))

    print(f"[Checkpoint saved to Drive] Epoch {trainer.epoch}")

model.add_callback("on_train_epoch_end", save_to_drive_callback)

model.train(
    data="/content/dataset_local/License-Plate-Recognition-12/data_final.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    name="license_plate_detector",
    project="/content/outputs"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.149 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_local/License-Plate-Recognition-12/data

Exception in thread Thread-9 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/python3.13/multiprocessing/resource

KeyboardInterrupt: 

In [11]:
import shutil, os

if os.path.exists("/content/outputs/license_plate_detector"):
    shutil.rmtree("/content/outputs/license_plate_detector")

drive_weights = "/content/drive/MyDrive/license_plate_project/weights"
for f in ["best.pt", "last.pt"]:
    path = os.path.join(drive_weights, f)
    if os.path.exists(path):
        os.remove(path)

print("Cleaned up old files")

Cleaned up old files


In [12]:
from ultralytics import YOLO
import shutil, os

model = YOLO("yolov8n.pt")

DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/license_plate_project/weights"
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)

def save_to_drive_callback(trainer):
    save_dir = trainer.save_dir
    best_path = os.path.join(save_dir, "weights", "best.pt")
    last_path = os.path.join(save_dir, "weights", "last.pt")
    if os.path.exists(best_path):
        shutil.copy(best_path, os.path.join(DRIVE_WEIGHTS_DIR, "best.pt"))
    if os.path.exists(last_path):
        shutil.copy(last_path, os.path.join(DRIVE_WEIGHTS_DIR, "last.pt"))
    print(f"[Checkpoint saved to Drive] Epoch {trainer.epoch}")

model.add_callback("on_train_epoch_end", save_to_drive_callback)

model.train(
    data=f"{dataset.location}/data_final.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    name="license_plate_detector_v2",
    project="/content/outputs"
)

Ultralytics 8.4.149 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_local/License-Plate-Recognition-12/data_final.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=license_p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fd77abffb60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [13]:
import os
weights_dir = "/content/drive/MyDrive/license_plate_project/weights"
print(os.listdir(weights_dir))

['best.pt', 'last.pt']


In [14]:
import cv2
import numpy as np
import os
import shutil
testImgDir=f"{dataset.location}/test/images"
normalDir="/content/test_normal/images"
edgeDir="/content/test_edge/images"
os.makedirs(normalDir, exist_ok=True)
os.makedirs(edgeDir, exist_ok=True)

normal_count=0
edge_count=0
for fname in os.listdir(testImgDir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png'))):
      continue
    img = cv2.imread(os.path.join(testImgDir, fname))
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    blur = cv2.Laplacian(gray, cv2.CV_64F).var()

    is_edge = (brightness < 90) or (blur < 100)

    if is_edge:
        shutil.copy(os.path.join(testImgDir, fname), os.path.join(edgeDir, fname))
        edge_count += 1
    else:
        shutil.copy(os.path.join(testImgDir, fname), os.path.join(normalDir, fname))
        normal_count += 1

print(f"Normal test images: {normal_count}")
print(f"Edge case test images: {edge_count}")

Normal test images: 723
Edge case test images: 297


In [15]:
from ultralytics import YOLO
base_model_path=f"/content/outputs/license_plate_detector_v2/weights/best.pt"
model=YOLO(base_model_path)
print("=== Testing on NORMAL images ===")
results_normal=model.predict(source=normalDir, conf=0.25, save=True, project="/content/outputs", name="test_normal_results")
print("\n=== Testing on EDGE CASE images ===")
results_edge=model.predict(source=edgeDir, conf=0.25, save=True, project="/content/outputs", name="test_edge_results")
print("\nDone!")

=== Testing on NORMAL images ===

image 1/723 /content/test_normal/images/0002a5b67e5f0909_jpg.rf.c8f81ef986e3e99af6f349c200080453.jpg: 480x640 2 License_Plates, 5.9ms
image 2/723 /content/test_normal/images/000812dcf304a8e7_jpg.rf.ba32e6c184b3d974abcced6f7c29af6d.jpg: 576x640 1 License_Plate, 76.4ms
image 3/723 /content/test_normal/images/0010f4c10f7ab07e_jpg.rf.1844f6dde3b97ed1c762db933bbacaf3.jpg: 480x640 1 License_Plate, 13.2ms
image 4/723 /content/test_normal/images/001cdd25e148cd36_jpg.rf.3921d4ff1d51af107666bc7ef7bd45b1.jpg: 480x640 1 License_Plate, 11.1ms
image 5/723 /content/test_normal/images/002519f868563098_jpg.rf.29775b804909c1d042ce008c09f033e4.jpg: 448x640 1 License_Plate, 75.5ms
image 6/723 /content/test_normal/images/0026c246d5c33bea_jpg.rf.e9b4d443d16831c5edd576804bb54beb.jpg: 448x640 2 License_Plates, 15.3ms
image 7/723 /content/test_normal/images/002901d9d194c4fb_jpg.rf.ba35ca49985646e7c1e01fb118f038d8.jpg: 480x640 2 License_Plates, 14.9ms
image 8/723 /content/test_

In [16]:
!pip install gradio pyngrok

import gradio as gr
from ultralytics import YOLO
import cv2

model = YOLO("/content/outputs/license_plate_detector_v2/weights/best.pt")

def detect_plate(image):
    results = model.predict(image, conf=0.25)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return annotated_rgb

demo = gr.Interface(
    fn=detect_plate,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Image(type="numpy"),
    title="License Plate Detection",
    description="Upload an image (normal or edge-case: blurry/night/angled) to detect license plates."
)

from pyngrok import ngrok
ngrok.set_auth_token("3JELEXhB6ftyPTR1nxMKs1PUij4_78D9AGCKrJ7KR2udrYJcP")
public_url = ngrok.connect(7860)
print("Public URL:", public_url)

demo.launch(server_port=7860)

Public URL: NgrokTunnel: "https://gathering-unsolved-slogan.ngrok-free.dev" -> "http://localhost:7860"
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b5921fd2b36d34c62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
